# GBIF Verification of occurence names
This notebook comes after running the models and obtaining on list of potential specie names

## Overview
We want to check if the names found are actual species names and retrieve a dictionnary of couple specie name / gbif link and a list of rejected specie names

In [2]:
import pandas as pd

# I - Installation of pygbif (Python for GBIF)

This python library is going to easily allow us to do the manipulation we need on the gbif backbone taxonomy

In [7]:
pip install pygbif

^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Users\romai\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


# II - Basic functions provided

In [3]:
from pygbif import species

Let's do an example with Canis lupus, which refers to the basic wolf specie, known as Canis lupus Linnaeus, 1758

In [12]:
name1 = "Canis lupus"
name2 = "Canus lupus"
name3 = "Canos lps"

We have a function that can return a lot of useful information with just a name from a string

In [7]:
res = species.name_backbone(scientificName=name1)
print(res)

{'usage': {'key': '5219173', 'name': 'Canis lupus Linnaeus, 1758', 'canonicalName': 'Canis lupus', 'authorship': 'Linnaeus, 1758', 'rank': 'SPECIES', 'code': 'ZOOLOGICAL', 'status': 'ACCEPTED', 'genericName': 'Canis', 'specificEpithet': 'lupus', 'type': 'SCIENTIFIC', 'formattedName': '<i>Canis</i> <i>lupus</i> Linnaeus, 1758'}, 'classification': [{'key': '1', 'name': 'Animalia', 'rank': 'KINGDOM'}, {'key': '44', 'name': 'Chordata', 'rank': 'PHYLUM'}, {'key': '359', 'name': 'Mammalia', 'rank': 'CLASS'}, {'key': '732', 'name': 'Carnivora', 'rank': 'ORDER'}, {'key': '9701', 'name': 'Canidae', 'rank': 'FAMILY'}, {'key': '5219142', 'name': 'Canis', 'rank': 'GENUS'}, {'key': '5219173', 'name': 'Canis lupus', 'rank': 'SPECIES'}], 'diagnostics': {'matchType': 'EXACT', 'confidence': 99, 'timeTaken': 2, 'timings': {'nameNRank': 0, 'sciNameMatch': 3, 'nameParse': 0, 'luceneMatch': 3}}, 'additionalStatus': [{'clbDatasetKey': '53131', 'datasetAlias': 'IUCN', 'datasetKey': '19491596-35ae-4a91-9a98-8

In the case of errors : 

In [10]:
res_false1 = species.name_backbone(scientificName=name2)
print(res_false1)

{'usage': {'key': '5219173', 'name': 'Canis lupus Linnaeus, 1758', 'canonicalName': 'Canis lupus', 'authorship': 'Linnaeus, 1758', 'rank': 'SPECIES', 'code': 'ZOOLOGICAL', 'status': 'ACCEPTED', 'genericName': 'Canis', 'specificEpithet': 'lupus', 'type': 'SCIENTIFIC', 'formattedName': '<i>Canis</i> <i>lupus</i> Linnaeus, 1758'}, 'classification': [{'key': '1', 'name': 'Animalia', 'rank': 'KINGDOM'}, {'key': '44', 'name': 'Chordata', 'rank': 'PHYLUM'}, {'key': '359', 'name': 'Mammalia', 'rank': 'CLASS'}, {'key': '732', 'name': 'Carnivora', 'rank': 'ORDER'}, {'key': '9701', 'name': 'Canidae', 'rank': 'FAMILY'}, {'key': '5219142', 'name': 'Canis', 'rank': 'GENUS'}, {'key': '5219173', 'name': 'Canis lupus', 'rank': 'SPECIES'}], 'diagnostics': {'matchType': 'VARIANT', 'confidence': 85, 'timeTaken': 2, 'timings': {'nameNRank': 0, 'sciNameMatch': 3, 'nameParse': 0, 'luceneMatch': 3}}, 'additionalStatus': [{'clbDatasetKey': '53131', 'datasetAlias': 'IUCN', 'datasetKey': '19491596-35ae-4a91-9a98

In [13]:
res_false2 = species.name_backbone(scientificName=name3)
print(res_false2)

{'diagnostics': {'matchType': 'NONE', 'issues': [], 'confidence': 100, 'timeTaken': 1, 'timings': {'sciNameMatch': 2}}, 'synonym': False}


In [6]:
res_false3 = species.name_backbone(scientificName="Canis Lupus")
res_false4 = species.name_backbone(scientificName="canis lupus")

print(res_false3)
print(res_false4)

{'diagnostics': {'matchType': 'NONE', 'issues': [], 'confidence': 100, 'note': 'No match because of too little confidence', 'timeTaken': 31, 'timings': {'sciNameMatch': 32}}, 'synonym': False}
{'usage': {'key': '5219173', 'name': 'Canis lupus Linnaeus, 1758', 'canonicalName': 'Canis lupus', 'authorship': 'Linnaeus, 1758', 'rank': 'SPECIES', 'code': 'ZOOLOGICAL', 'status': 'ACCEPTED', 'genericName': 'Canis', 'specificEpithet': 'lupus', 'type': 'SCIENTIFIC', 'formattedName': '<i>Canis</i> <i>lupus</i> Linnaeus, 1758'}, 'classification': [{'key': '1', 'name': 'Animalia', 'rank': 'KINGDOM'}, {'key': '44', 'name': 'Chordata', 'rank': 'PHYLUM'}, {'key': '359', 'name': 'Mammalia', 'rank': 'CLASS'}, {'key': '732', 'name': 'Carnivora', 'rank': 'ORDER'}, {'key': '9701', 'name': 'Canidae', 'rank': 'FAMILY'}, {'key': '5219142', 'name': 'Canis', 'rank': 'GENUS'}, {'key': '5219173', 'name': 'Canis lupus', 'rank': 'SPECIES'}], 'diagnostics': {'matchType': 'EXACT', 'confidence': 99, 'timeTaken': 2, 't

We can understand that we have different matchType status being EXACT, VARIANT and NONE => definiton of our acceptance of mispelling 

Now to obtain the website link from the function

We observe that in the usage case we can find a key which can be used to link to the specie page

In [16]:
print(res.get("usage"))
key = res["usage"]["key"]
print(key)

print(f"GBIF taxonomy page: https://www.gbif.org/species/{key}")

{'key': '5219173', 'name': 'Canis lupus Linnaeus, 1758', 'canonicalName': 'Canis lupus', 'authorship': 'Linnaeus, 1758', 'rank': 'SPECIES', 'code': 'ZOOLOGICAL', 'status': 'ACCEPTED', 'genericName': 'Canis', 'specificEpithet': 'lupus', 'type': 'SCIENTIFIC', 'formattedName': '<i>Canis</i> <i>lupus</i> Linnaeus, 1758'}
5219173
GBIF taxonomy page: https://www.gbif.org/species/5219173


# III - Fonction de vérification

In [4]:
def verif(names_list) :
    correct_occurences = {}
    misspelled_ocrurrences = []
    wrong_occurences = []
    # We initialize one dictionnary for the correct specie names with
    # the gbif link, and two other lists for variant spellings and
    # false occurences


    # On itère sur toute la liste de noms et on vérifie le status de
    # la recherche pour décider dans quelle liste ajouter le nom
    for name in names_list : 
        res = species.name_backbone(scientificName=name)
        print("Testing" + name)
        if res["diagnostics"]["matchType"] == "EXACT" :
            correct_occurences[res["usage"]["canonicalName"]] = f"GBIF taxonomy page - https://www.gbif.org/species/{res["usage"]["key"]}"
        elif res["diagnostics"]["matchType"] == "VARIANT" :
            misspelled_ocrurrences.append([name, res["usage"]["canonicalName"], f"GBIF taxonomy page - https://www.gbif.org/species/{res["usage"]["key"]}"])
        else : 
            wrong_occurences.append(name)

    return(correct_occurences, misspelled_ocrurrences, wrong_occurences)
    

Ainsi, on obtient en sortie le dictionnaire avec la liste de noms correctement identifiés et leur lien, une liste de noms trouvés de manière approximative par le site gbif avec le vrai nom de l'espèce et le nom relevé par les modèles pour pouvoir effectuer une comparaison après, et une liste de noms rejetés

Essai manuel

In [34]:
test = ["Canis lupus", "Equus ferus", "Equus Caballus", "Equus caballus", "Canus lupos", "Canus lps"]

In [35]:
a, b, c = verif(test)

In [36]:
print(a)
print(b)
print(c)

{'Canis lupus': 'GBIF taxonomy page: https://www.gbif.org/species/5219173', 'Equus ferus': 'GBIF taxonomy page: https://www.gbif.org/species/4409270', 'Equus caballus': 'GBIF taxonomy page: https://www.gbif.org/species/2440886'}
[['Canus lupos', 'Canis lupus', 'GBIF taxonomy page: https://www.gbif.org/species/5219173']]
['Equus Caballus', 'Canus lps']


In [15]:
print(verif(["Ostrea edulis"]))

({'Ostrea edulis': 'GBIF taxonomy page - https://www.gbif.org/species/2286060'}, [], [])


On peut déjà noter que le capitalisation pose problème et un nom correct mais mal capitalisé passe comme non détecté

In [1]:
def result_csv(csv_path) :
    csv = pd.read_csv(csv_path)
    csv["Accepted Names"] = "couldn't find"
    csv["Misspelled Names"] = ""
    csv["Unrecognised Names"] = ""
    length = len(csv)
    for i in range(length) : 
        if csv.at[i, "Taxon"] != "couldn't find" :
            strings = [x.strip() for x in csv.at[i, "Taxon"].strip("[]").split(",")] 
            a, b, c = verif(strings)
            csv.at[i, "Accepted Names"] = str(a)
            csv.at[i, "Misspelled Names"] = str(b)
            csv.at[i, "Unrecognised Names"] = str(c)
    csv.to_csv("result_checked.csv")

In [5]:
result_csv("result.csv")

Testing'Ostrea edulis'
Testing'Ostrea edulis'
Testing'Epinephelus marginatus'
Testing'Sciaena umbra'
Testing'Diplodus cervinus'
Testing'Lyallia kerguelensis'
Testing'Lyallia kerguelensis'
Testing'Lyallia kerguelensis'
Testing'Lyallia kerguelensis'
Testing'Trisopterus minutus'
Testing'Trisopterus minutus'
Testing'Capreolus capreolus'
Testing'Cervus elaphus'
Testing'Ovis gmelini'
Testing'Rupicapra rupicapra'
Testing'Grands puffins'
Testing'Puffinus gravis'
Testing'Puffinus griseus'
Testing'Avena fatua'
Testing'Lolium multiflorum'
Testing'Fallopia convolvulus'
Testing'Galium aparine'
Testing'Anisantha sterilis'
Testing'Zostera marina'
Testing'Zostera noltii'
Testing'Pholas dactilus'
Testing'Cerastoderma edule'
Testing'Mimachlamys varia'
Testing'Chlamys varia'
Testing'Crassostrea angulata'
Testing'Mytilus edulis'
Testing'Himanthalia elongata'
Testing'Himanthalia lorea'
Testing'Chondrus crispus'
Testing'Pertuis charentais'
Testing'Pertuis charentais'
Testing'Zostera marina'
Testing'Mytilus 